Intent de entrenament d'un model que predigui el millor possible un paràmetre

In [1]:
print("Hola")

Hola


In [2]:
import os
import sys
print(os.getcwd())

sys.path.append(r'c:\Users\jmass\Documents\GitHub\TFG_GEINF\src\data')
from DatasetGenerator import DatasetGenerator


c:\Users\jmass\Documents\GitHub\TFG_GEINF\notebooks\experiments


In [3]:
import pandas as pd

num_particules = 10000

dg = DatasetGenerator(num_particules)

In [4]:
data = dg.generate_BrownianMotion(random_params=True, num_passos=200)

In [5]:
data["attributes"][0]["sigma"]

1.001555179708282

In [6]:
df = data[["coords", "attributes"]].copy()
df["sigma"] = df["attributes"].apply(lambda x: x["sigma"])
df = df[["coords", "sigma"]]

In [7]:
df

,coords,sigma
0,"[[0.0, 0.15958454942957065, -1.038931564786983...",1.001555
1,"[[0.0, 0.11280472853100115, 0.1812704875805188...",0.128205
2,"[[0.0, 1.6086845590271437, 2.797489373131696, ...",2.633431
3,"[[0.0, -1.6859600495944906, -3.680344787949861...",2.092048
4,"[[0.0, 0.2863845099222955, 1.6505276525906756,...",2.617002
...,...,...
9995,"[[0.0, -0.14970761051002252, 0.001626630946373...",0.707425
9996,"[[0.0, -2.7017345459098245, 0.2184230810077974...",2.492666
9997,"[[0.0, -0.6178043201916487, 0.75747832415186, ...",2.294013
9998,"[[0.0, -2.7149423703070865, -5.066519537477903...",1.887582


In [8]:
print(df.columns)

Index(['coords', 'sigma'], dtype='object')


In [9]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

In [12]:
# 1. Convertir las coordenadas a tensor 3D
# Asumiendo que 'data' es tu DataFrame
coords_list = [np.stack([row[0], row[1]], axis=1) for row in df['coords']]
X = np.array(coords_list)  # Forma (n_samples, seq_length, 2)
X_tensor = torch.tensor(X, dtype=torch.float32)

In [ ]:
# 2. Convertir las etiquetas a tensor de valores reales
df["sigma"] = df["sigma"].apply(lambda x: (x - df["sigma"].min()) / (df["sigma"].max() - df["sigma"].min()))
labels = df['sigma'].values  # Suponiendo que 'sigma' contiene los valores reales
y_tensor_temp = torch.tensor(labels, dtype=torch.float32)

In [14]:
df["sigma"]

0      -0.378238
1      -0.980548
2       0.747194
3       0.373826
4       0.735864
          ...   
9995   -0.581086
9996    0.650115
9997    0.513112
9998    0.232815
9999   -0.688012
Name: sigma, Length: 10000, dtype: float64

In [15]:
y_tensor = y_tensor_temp.unsqueeze(1)  # Agrega una dimensión adicional
y_tensor.shape

torch.Size([10000, 1])

In [16]:
y_tensor

tensor([[-0.3782],
        [-0.9805],
        [ 0.7472],
        ...,
        [ 0.5131],
        [ 0.2328],
        [-0.6880]])

In [17]:
# 4. Dividir datos en entrenamiento y prueba (ejemplo simple)
train_size = int(0.8 * len(X_tensor))
X_train, X_test = X_tensor[:train_size], X_tensor[train_size:]
y_train, y_test = y_tensor[:train_size], y_tensor[train_size:]

In [18]:
# 5. Crear DataLoader
batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# 6. Definir la arquitectura CNN
class TrajectoryCNN(nn.Module):
    def __init__(self, input_channels, seq_length):
        super().__init__()
        self.first_conv = nn.Conv1d(input_channels, 32, kernel_size=3, padding=1)
        self.conv1_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv2_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv_32_64 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv1_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv2_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv_64_128 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.conv1_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.conv2_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.max_pool = nn.MaxPool1d(2)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 4))
        self.fc1 = nn.Linear(4, 256)
        self.fc2 = nn.Linear(256, 1)

    
    def forward(self, data):
        loop = 2
        a_resnet = self.first_conv(data)
        a = self.conv1_32(a_resnet)
        a = self.conv2_32(a)
        a = a + a_resnet
        a = self.max_pool(a)
        b_resnet = self.conv_32_64(a)
        b = self.conv1_64(b_resnet)
        b = self.conv2_64(b)
        b = b + b_resnet
        b = self.max_pool(b)
        c_resnet = self.conv_64_128(b)
        c = self.conv1_128(c_resnet)
        c = self.conv2_128(c)
        c = c + c_resnet
        c = self.avg_pool(c)
        c = c.view(c.size(0), -1)  # Aplanar la salida
        c = F.relu(self.fc1(c))
        c = self.fc2(c)
        c = torch.clamp(c, min=-1, max=1)
        return c

In [ ]:
import torch
import torch.nn as nn

# Parámetros del ejemplo
input_size = 2       # Por ejemplo, coordenadas x, y
hidden_size = 64     # Número de neuronas en la capa LSTM
num_layers = 3       # Capas LSTM apiladas
output_size = 1      # Valor escalar a predecir

# Modelo LSTM para regresión
class LSTMRegressor(nn.Module):
    def __init__(self):
        super(LSTMRegressor, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),  # entrada: 1 canal
            nn.ReLU(),
            nn.MaxPool2d(2),  # 14x14
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)   # 7x7
        )

    def forward(self, x):
        out, _ = self.lstm(x)  # out: (batch_size, seq_length, hidden_size)
        out = out[:, -1, :]    # tomamos la última salida de la secuencia
        out = self.fc(out)     # regresión final
        out = torch.clamp(out, min=-1, max=1)
        return out



In [20]:
# Crear el modelo
model = LSTMRegressor()

In [21]:
# model.first_conv.weight.shape

In [22]:
# 8. Configurar entrenamiento
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [23]:
# 9. Entrenar el modelo
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss/len(train_loader):.4f}")


Epoch 1/10, Loss: 0.1989
Epoch 2/10, Loss: 0.1604
Epoch 3/10, Loss: 0.1393
Epoch 4/10, Loss: 0.1311
Epoch 5/10, Loss: 0.1269
Epoch 6/10, Loss: 0.1222
Epoch 7/10, Loss: 0.1185
Epoch 8/10, Loss: 0.1175
Epoch 9/10, Loss: 0.1159
Epoch 10/10, Loss: 0.1140


In [24]:
# 10. Evaluación (opcional)
with torch.no_grad():
    test_outputs = model(X_test).squeeze()  # Asegurarse de que las dimensiones coincidan
    mse = criterion(test_outputs, y_test)  # Calcular el MSE
    print(f'Test MSE: {mse.item():.4f}')


Test MSE: 0.5537


c:\Users\jmass\Documents\GitHub\TFG_GEINF\env\Lib\site-packages\torch\nn\modules\loss.py:610: UserWarning: Using a target size (torch.Size([2000, 1])) that is different to the input size (torch.Size([2000])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [44]:
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [46]:
# Evaluación
model.eval()  # modo evaluación
total_loss = 0.0
all_preds = []
all_targets = []

In [47]:
with torch.no_grad():  # desactiva cálculo de gradientes
    for x_batch, y_batch in test_loader:
        preds = model(x_batch)
        loss = criterion(preds, y_batch)
        total_loss += loss.item()

        all_preds.append(preds)
        all_targets.append(y_batch)

# Promedio de pérdida
avg_loss = total_loss / len(test_loader)
print(f"Test Loss: {avg_loss:.4f}")


Test Loss: 0.7044


c:\Users\jmass\Documents\GitHub\TFG_GEINF\env\Lib\site-packages\torch\nn\modules\loss.py:610: UserWarning: Using a target size (torch.Size([16])) that is different to the input size (torch.Size([16, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [ ]:
# Guardar solo los parámetros del modelo
torch.save(model.state_dict(), 'model_BrownianMotion_sigma_prediction.pth')

# Si deseas guardar el modelo completo
torch.save(model, 'model_BrownianMotion_sigma_prediction_complete.pth')

In [ ]:
# Cargar solo los parámetros del modelo
model = TrajectoryCNN(
    input_channels=2,
    seq_length=X_tensor.shape[2],  # m (longitud de la secuencia)
    num_classes=len(le.classes_)
)
model.load_state_dict(torch.load('trajectory_cnn_model_2.pth'))
model.eval()

# Si deseas cargar el modelo completo
# model = torch.load('trajectory_cnn_model_complete.pth')
# model.eval()